# 3. Data Preparation
We apply the decisions from Data Understanding, one step at a time.
Missing-value replacements, scaling and category frequencies are learned
from development purchases only.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Load the original files
This notebook starts from raw data and can run independently.

In [2]:
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
clientes = pd.read_csv("../data/raw/clientes.csv")
target = "repeat_purchase_90d"

print("Train:", train.shape)
print("Test:", test.shape)

Train: (6534, 39)
Test: (986, 38)


## Remove exact duplicates
We compare all fields except the unique purchase ID, before dropping columns.

In [3]:
cols = train.columns.drop("ID")
dup = train.duplicated(subset=cols)
print("Exact duplicates removed:", dup.sum())
train = train.loc[~dup].copy()

Exact duplicates removed: 10


## Dates and incomplete purchases
The 55 distinct undated records have no purchase information. We keep them
in `undated` for inspection and leave them out of the chronological split.
Their source records remain in the raw file.

In [4]:
train["purchase_date"] = pd.to_datetime(train["purchase_date"])
test["purchase_date"] = pd.to_datetime(test["purchase_date"])

undated = train.loc[train["purchase_date"].isna()].copy()
train = train.loc[train["purchase_date"].notna()].copy()

print("Undated rows excluded:", len(undated))
print("Dated training rows:", len(train))
print("Test rows without a date:", test["purchase_date"].isna().sum())

Undated rows excluded: 55
Dated training rows: 6469
Test rows without a date: 0


## Invalid history values
We replace negatives only in the four fields identified during exploration.
Other negative values, including `random_noise`, are preserved.

In [5]:
cols = ["days_since_previous_purchase", "prior_mean_gap_days",
        "prior_mean_purchase_value", "spend_365d"]

for col in cols:
    train.loc[train[col] < 0, col] = np.nan
    test.loc[test[col] < 0, col] = np.nan

The repeated exact value 1,000,000,000 is a suspected placeholder in the
following two columns. This is a cleaning assumption, not an official missing-value code.

In [6]:
cols = ["prior_mean_purchase_value", "spend_365d"]
for col in cols:
    train.loc[train[col] == 1_000_000_000, col] = np.nan
    test.loc[test[col] == 1_000_000_000, col] = np.nan

## Reconstruct historical averages
A historical mean equals total historical spend divided by the number
of previous purchases. Exploration showed this relationship holds
for almost all observed averages. We use it to replace inconsistent
averages and fill missing averages when previous purchases exist.

This assumes the supplied totals and counts are reliable. For first
purchases, count is zero and the historical mean remains missing.
The 0.01 tolerance allows rounding differences.

In [7]:
for name, df in [("train", train), ("test", test)]:
    mean = df["prior_total_spend"] / df["prior_purchase_count"].replace(0, np.nan)
    bad = (df["prior_mean_purchase_value"] - mean).abs() > 0.01
    print(name, "inconsistent historical means:", bad.sum())
    df.loc[bad, "prior_mean_purchase_value"] = np.nan
    df["prior_mean_purchase_value"] = df["prior_mean_purchase_value"].fillna(mean)

train inconsistent historical means: 46
test inconsistent historical means: 7


## Remove inconsistent history values
A maximum purchase cannot exceed all historical spend. Annual spend
must lie between the last 180-day spend and all-time spend.
We mark violations as missing values and keep the purchase rows.
Coherent high values are retained, regardless of their size.

In [8]:
for name, df in [("train", train), ("test", test)]:
    bad = df["prior_max_purchase_value"] > df["prior_total_spend"] + 0.01
    print(name, "inconsistent historical maxima:", bad.sum())
    df.loc[bad, "prior_max_purchase_value"] = np.nan

train inconsistent historical maxima: 45
test inconsistent historical maxima: 6


In [9]:
for name, df in [("train", train), ("test", test)]:
    low = df["spend_365d"] < df["spend_180d"] - 0.01
    high = df["spend_365d"] > df["prior_total_spend"] + 0.01
    print(name, "inconsistent annual spend:", (low | high).sum())
    df.loc[low | high, "spend_365d"] = np.nan

train inconsistent annual spend: 39
test inconsistent annual spend: 7


## Add customer information
Customer IDs are unique in `clientes`, so the left join adds attributes
without multiplying purchases.

In [10]:
train = train.merge(clientes, on="customer_id", how="left", validate="many_to_one")
test = test.merge(clientes, on="customer_id", how="left", validate="many_to_one")

print("Train after joining:", train.shape)
print("Test after joining:", test.shape)

Train after joining: (6469, 51)
Test after joining: (986, 50)


`customer_type` is empty. Company name and address are identifying. Street
and postal code are shared geographic categories; we initially exclude
these granular fields because of their cardinality and coverage. Broader
geography is retained. Their inclusion can be assessed during feature selection.

In [11]:
cols = ["customer_type", "company_name", "address", "street", "postal_code"]
train = train.drop(columns=cols)
test = test.drop(columns=cols)

## Remove variables without a relationship to the target
In Data Understanding, `random_noise` and `Total Portes` showed no relationship
with the target in the ANOVA, Mann-Whitney and mutual information tests.
`random_noise` is also supplied as noise. We remove both before modelling.

In [12]:
cols = ["random_noise", "Total Portes"]
train = train.drop(columns=cols)
test = test.drop(columns=cols)

## Keep clean inputs for later fitting
We retain the cleaned, unscaled data before splitting. Modeling will refit
missing-value handling, scaling and encoding inside each training fold;
Deployment will refit them on all eligible training purchases.

In [13]:
os.makedirs("../data/clean", exist_ok=True)
train.to_csv("../data/clean/train.csv", index=False)
test.to_csv("../data/clean/test.csv", index=False)

## Chronological split
We reserve the latest approximately 20% for validation, keeping each date
together. The 90-day gap gives development targets time to become known.
Repeated customers may appear in both periods.

In [14]:
train = train.sort_values(["purchase_date", "ID"]).reset_index(drop=True)
cut = train["purchase_date"].quantile(0.8).normalize()
gap = cut - pd.Timedelta(days=90)

valid = train.loc[train["purchase_date"] >= cut].copy()
buffer = train.loc[train["purchase_date"].between(gap, cut, inclusive="left")].copy()
train = train.loc[train["purchase_date"] < gap].copy()

In [15]:
print("Development:", len(train), "purchases")
print("90-day gap:", len(buffer), "purchases")
print("Validation:", len(valid), "purchases")
print("Test:", len(test), "purchases")
print("\nLast development purchase:", train["purchase_date"].max().date())
print("Validation starts:", cut.date())

Development: 5024 purchases
90-day gap: 147 purchases
Validation: 1298 purchases
Test: 986 purchases

Last development purchase: 2023-03-18
Validation starts: 2023-06-17


The split gives 5,024 development purchases, 147 in the gap and 1,298 for
validation. All 986 test purchases remain in their original order.

## Calendar variables
Month and weekday are available at the current purchase and are treated as
categories. Since dates are anonymised, their predictive value is provisional
and must be assessed later.

We do not use the purchase year. Test purchases fall in 2025 and 2026, after
every training purchase. If the model learns, for example, that there were more
repeat purchases in 2024, this does not help at all with predictions in 2026,
a year it has never seen.

In [16]:
for df in [train, valid, test]:
    df["purchase_month"] = df["purchase_date"].dt.month.astype(str)
    df["purchase_weekday"] = df["purchase_date"].dt.dayofweek.astype(str)

## Separate inputs and target
We use purchase ID as the index for tracking. The ID, customer ID, original
date and target are excluded from the model inputs.

In [17]:
train = train.set_index("ID")
valid = valid.set_index("ID")
test = test.set_index("ID")

y_train = train[target].copy()
y_valid = valid[target].copy()

cols = ["customer_id", "purchase_date", target]
X_train = train.drop(columns=cols)
X_valid = valid.drop(columns=cols)
X_test = test.drop(columns=["customer_id", "purchase_date"])

In [18]:
num = X_train.select_dtypes(include="number").columns.tolist()
cat = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numeric variables:", len(num))
print("Categorical variables:", len(cat))
X_train.head()

Numeric variables: 28
Categorical variables: 14


,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,days_since_previous_purchase,...,document_series,country_or_region,commercial_region,island,municipality,city,postal_area,customer_since_year,purchase_month,purchase_weekday
ID,,,,,,,,,,,,,,,,,,,,,
10000812,55.86,51.99,51.99,3.87,0.00,0.00,28.00,1.0,1.0,NaN,...,Ctrl Z Series 3,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU5: Council of Last Minute Edits,NaN,Postal area PA269: Hidden Sheet Sector,2015,7,5
10007082,268.76,436.99,218.50,50.26,218.49,0.00,116.14,9.0,1.0,NaN,...,Ctrl Z Series 3,Country CT10: Land of The Final Version,NaN,NaN,NaN,NaN,Postal area PA61: Hidden Sheet Sector,2015,7,5
10000232,541.92,508.20,508.20,33.72,0.00,0.00,285.70,6.0,1.0,NaN,...,Ctrl Z Series 3,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS5: The Pivot Archipelago,Municipality MU17: Council of Last Minute Edits,NaN,Postal area PA252: Hidden Sheet Sector,2015,7,6
10000233,1752.09,1552.61,1424.46,327.63,0.00,128.15,402.01,46.0,1.0,NaN,...,Ctrl Z Series 3,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS6: The Pivot Archipelago,Municipality MU2: Council of Last Minute Edits,City CY59: AutoFill Harbor,Postal area PA262: Hidden Sheet Sector,2015,7,6
10000524,76.97,62.58,62.58,14.39,0.00,0.00,35.70,4.0,1.0,NaN,...,Ctrl Z Series 3,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS3: The Pivot Archipelago,Municipality MU13: Council of Last Minute Edits,NaN,Postal area PA264: Hidden Sheet Sector,2015,7,6


## Missing-value flags
For each numeric column with missing development values, we add a binary
flag before filling the gaps. This preserves the difference between an
observed value and an imputed value, including unavailable purchase history.

In [19]:
for col in num:
    if X_train[col].isna().any():
        X_train[col + "_missing"] = X_train[col].isna().astype(int)
        X_valid[col + "_missing"] = X_valid[col].isna().astype(int)
        X_test[col + "_missing"] = X_test[col].isna().astype(int)

## Fill numeric gaps
All medians come from development data. Validation and test use the same
values; neither contributes to the calculation.

In [20]:
med = X_train[num].median()
X_train[num] = X_train[num].fillna(med)
X_valid[num] = X_valid[num].fillna(med)
X_test[num] = X_test[num].fillna(med)

print("Remaining numeric gaps in development:", X_train[num].isna().sum().sum())

Remaining numeric gaps in development: 0


## Fill categorical gaps
Missing text values become an explicit category.

In [21]:
for col in cat:
    X_train[col] = X_train[col].fillna("Missing")
    X_valid[col] = X_valid[col].fillna("Missing")
    X_test[col] = X_test[col].fillna("Missing")

## Reduce monetary skewness
Coherent high amounts are retained. `log1p` compresses the monetary
tails and keeps zeros valid. It is applied only to non-negative
monetary fields, not to dates or counts.

In [22]:
money = ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_",
         "Total Desc_ Global", "Total Desc_ Linha", "Custo",
         "prior_total_spend", "prior_mean_purchase_value",
         "prior_min_purchase_value", "prior_max_purchase_value",
         "prior_std_purchase_value", "spend_30d", "spend_90d",
         "spend_180d", "spend_365d"]

X_train[money] = np.log1p(X_train[money])
X_valid[money] = np.log1p(X_valid[money])
X_test[money] = np.log1p(X_test[money])

## Scale numeric variables
`StandardScaler` centres each numeric input and scales it by its
development standard deviation. Validation and test use the same
parameters. Binary missing-value flags stay 0/1.

This also scales the two fields whose interquartile range is zero
but whose standard deviation is positive. Combined with the monetary
log transform, this gives us a consistent starting representation.

In [23]:
scaler = StandardScaler()
X_train[num] = scaler.fit_transform(X_train[num])
X_valid[num] = scaler.transform(X_valid[num])
X_test[num] = scaler.transform(X_test[num])

X_train[num].head().round(2)

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,days_since_previous_purchase,...,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_since_year
ID,,,,,,,,,,,,,,,,,,,,,
10000812,-1.43,-1.43,-1.34,-1.79,-0.11,-1.01,-0.64,-0.72,-0.22,-0.26,...,-0.2,-0.5,-0.69,-0.53,-1.04,-0.59,-1.71,-0.61,-2.25,-0.68
10007082,-0.28,0.13,-0.29,-0.15,17.18,-1.01,0.09,-0.42,-0.22,-0.26,...,-0.2,-0.5,-0.69,-0.53,-1.04,-0.59,-1.71,-0.61,-2.25,-0.68
10000232,0.24,0.24,0.33,-0.42,-0.11,-1.01,0.56,-0.53,-0.22,-0.26,...,-0.2,-0.5,-0.69,-0.53,-1.04,-0.59,-1.71,-0.61,-2.25,-0.68
10000233,1.11,1.07,1.09,1.14,-0.11,1.11,0.74,0.93,-0.22,-0.26,...,-0.2,-0.5,-0.69,-0.53,-1.04,-0.59,-1.71,-0.61,-2.25,-0.68
10000524,-1.20,-1.30,-1.20,-0.99,-0.11,-1.01,-0.51,-0.61,-0.22,-0.26,...,-0.2,-0.5,-0.69,-0.53,-1.04,-0.59,-1.71,-0.61,-2.25,-0.68


## Encode categories
The encoder learns from development data. Categories with fewer than 20
development purchases are grouped together. An unseen category uses that
group when available; otherwise its one-hot block is all zeros. This
threshold is provisional.

In [24]:
enc = OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist",
                    sparse_output=False)

cat_train = enc.fit_transform(X_train[cat])
cat_valid = enc.transform(X_valid[cat])
cat_test = enc.transform(X_test[cat])

In [25]:
cols = enc.get_feature_names_out(cat)
cat_train = pd.DataFrame(cat_train, columns=cols, index=X_train.index)
cat_valid = pd.DataFrame(cat_valid, columns=cols, index=X_valid.index)
cat_test = pd.DataFrame(cat_test, columns=cols, index=X_test.index)

X_train = X_train.drop(columns=cat).join(cat_train)
X_valid = X_valid.drop(columns=cat).join(cat_valid)
X_test = X_test.drop(columns=cat).join(cat_test)

## Review the prepared data
The three sets should have the same feature columns and no missing values.

In [26]:
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)
print("\nMissing values in train:", X_train.isna().sum().sum())
print("Missing values in validation:", X_valid.isna().sum().sum())
print("Missing values in test:", X_test.isna().sum().sum())
print("Same columns:", X_train.columns.equals(X_valid.columns)
      and X_train.columns.equals(X_test.columns))

X_train: (5024, 273)
X_valid: (1298, 273)
X_test: (986, 273)

Missing values in train: 0
Missing values in validation: 0
Missing values in test: 0
Same columns: True


In [27]:
X_train.iloc[:5, :8].round(2)

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes
ID,,,,,,,,
10000812,-1.43,-1.43,-1.34,-1.79,-0.11,-1.01,-0.64,-0.72
10007082,-0.28,0.13,-0.29,-0.15,17.18,-1.01,0.09,-0.42
10000232,0.24,0.24,0.33,-0.42,-0.11,-1.01,0.56,-0.53
10000233,1.11,1.07,1.09,1.14,-0.11,1.11,0.74,0.93
10000524,-1.20,-1.30,-1.20,-0.99,-0.11,-1.01,-0.51,-0.61


## Save the three prepared sets
We save one CSV per set. Training and validation include the target; all
files retain purchase ID for tracking. IDs and the target must be excluded
when fitting a classifier.

These CSVs are prepared for this fixed holdout. For cross-validation or
training on all available data, start from raw data and refit preparation
on each training fold or the full eligible training set.

In [28]:
os.makedirs("../data/prepared", exist_ok=True)

res = X_train.copy()
res[target] = y_train
res.to_csv("../data/prepared/train.csv", index=True)

res = X_valid.copy()
res[target] = y_valid
res.to_csv("../data/prepared/valid.csv", index=True)

X_test.to_csv("../data/prepared/test.csv", index=True)
print("Prepared files saved in data/prepared.")

Prepared files saved in data/prepared.


## Before modelling
Preparation is ready. Continue with [Modeling](04_Modeling.ipynb).

- Historical averages are reconstructed under the assumption that
  total spend and purchase count are reliable. Impossible maxima and
  annual spend are treated as missing, with flags before imputation.
- Median replacements are model inputs, not recovered transaction
  histories. Coherent large purchases have not been removed.
- `random_noise` and `Total Portes` are removed after the tests in Data
  Understanding, and the purchase year is not used. Month and weekday are
  candidates for L1 selection.
  The rare-category threshold stays at 20; alternative thresholds and
  calendar feature sets remain for the final project.
- The official Kaggle metric is **F1-Score**.

The three small helpers in `preparation.py` repeat the learned steps
shown above when fitting inner folds and the final model.